# $\phi$ \ $\chi$ spectra
Comparing observed spectra of acoustic phase, $\phi$ and log-amplitude, $\chi$ with modelled spectra given Garrett-Munk internal waves spectra (calculated with WFT).

In [ ]:
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from dotenv import find_dotenv, load_dotenv
from scipy import signal, optimize

from nescafe23_fluctuations import tools
import warnings
_ = load_dotenv(find_dotenv())

%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

## open observations and LS fit

In [ ]:
# open observations
m_data, F_phi_data, F_chi_data, F_phi_lm, F_phi_ls, F_chi_lm, F_chi_ls = tools.estimate_F_phi_chi()

In [ ]:
# open ls parameters for spectra + noise model
fn = f"{os.environ['NESCAFE23_DATADIR']}dataproducts/phi_chi_LS_fit.pkl"

with open(fn, 'rb') as f:
    results = pickle.load(f)

In [ ]:
ray_id = '+6'

F_phi_model, F_chi_model, F_phi_model_nonoise, F_chi_model_nonoise = tools.phi_chi_spectrum_model(m, results[ray_id].x[0], results[ray_id].x[1], results[ray_id].x[2], ray_id)


sns.set_palette('tab20')

# F_phi modeled
plt.plot(m[0]/(2*np.pi), F_phi_model, c='C0', lw=1, linestyle='dashed')
plt.plot(m[0]/(2*np.pi), F_phi_model_nonoise, c='C0', lw=2)

# F_phi observed
plt.plot(m_data, F_phi_lm[ray_id], c="C1", marker="o", markersize=3)
plt.fill_between(
    m_data,
    F_phi_lm[ray_id] * F_phi_ls[ray_id],
    F_phi_lm[ray_id] / F_phi_ls[ray_id],
    alpha=0.3,
    color="C1",
    lw=0,
)

# F_chi modeled
plt.plot(m[0]/(2*np.pi), F_chi_model, c='C2', lw=1, linestyle='dashed')
plt.plot(m[0]/(2*np.pi), F_chi_model_nonoise, c='C2', lw=2)

# F_chi observed
plt.plot(m_data, F_chi_lm[ray_id], c="C3", marker="o", markersize=3)
plt.fill_between(
    m_data,
    F_chi_lm[ray_id] * F_chi_ls[ray_id],
    F_chi_lm[ray_id] / F_chi_ls[ray_id],
    alpha=0.3,
    color="C3",
    lw=0,
)

plt.yscale('log')
plt.xscale('log')

plt.title(f'{ray_id}: {results[ray_id].x[0]:0.3f} $\\times$ GM energy', loc='left')

## Plot Spectra

In [ ]:
ray_id = '+7'

F_phi_model, F_chi_model, F_phi_model_nonoise, F_chi_model_nonoise = tools.phi_chi_spectrum_model(m, results[ray_id].x[0], results[ray_id].x[1], results[ray_id].x[2], '-8')
F_phi_model2, F_chi_model2, F_phi_model_nonoise2, F_chi_model_nonoise2 = tools.phi_chi_spectrum_model(m, results[ray_id].x[0], results[ray_id].x[1], results[ray_id].x[2], '+7')

In [ ]:
plt.plot(m[0], F_phi_model_nonoise)
plt.plot(m[0], F_phi_model_nonoise2)

plt.plot(m[0], F_chi_model_nonoise)
plt.plot(m[0], F_chi_model_nonoise2)

plt.xscale('log')
plt.yscale('log')

In [ ]:
plt.rcParams.update({"font.size": 8})
plt.rcParams["text.usetex"] = True
sns.set_palette("tab20")

fig, axes = plt.subplots(2, 2, figsize=(6.67, 5))
axes = axes.flatten()

for k, ray_id in enumerate(["+7", "-7", "+6", "-6"]):
    
    F_phi_model, F_chi_model, F_phi_model_nonoise, F_chi_model_nonoise = tools.phi_chi_spectrum_model(m, results[ray_id].x[0], results[ray_id].x[1], results[ray_id].x[2], ray_id)

    # compute model spectra for -8 as well (if +7 arrival)
    if ray_id == "+7":
        F_phi_model, F_chi_model, F_phi_model_nonoise, F_chi_model_nonoise = tools.phi_chi_spectrum_model(m, results[ray_id].x[0], results[ray_id].x[1], results[ray_id].x[2], '-8')

    # extend noise floor to the right
    nan_mask = np.isnan(F_phi_model)
    F_phi_model = np.hstack((F_phi_model[~nan_mask], F_phi_model[~nan_mask][-1]))
    F_chi_model = np.hstack((F_chi_model[~nan_mask], F_chi_model[~nan_mask][-1]))
    m_extend = np.concatenate((m[:,~nan_mask], np.array([[100]])), axis=1)
    
    plt.sca(axes[k])

    # WFT GMIW F_phi
    plt.plot(m[0] / (2 * np.pi), F_phi_model_nonoise, c="C0", lw=1.5, alpha=0.85)
    if ray_id == '+7':
        plt.plot(m[0] / (2 * np.pi), F_phi_model_nonoise2, c="C0", linestyle='dashdot', lw=1.5, alpha=0.85)
    plt.plot(m_extend[0] / (2 * np.pi), F_phi_model, c="C0", lw=0.75, linestyle='dashed')
    
    # F_phi observed
    plt.plot(m_data, F_phi_lm[ray_id], c="C1", marker="o", markersize=3)
    plt.fill_between(
        m_data,
        F_phi_lm[ray_id] * F_phi_ls[ray_id],
        F_phi_lm[ray_id] / F_phi_ls[ray_id],
        alpha=0.3,
        color="C1",
        lw=0,
    )

    # WFT GMIW F_chi
    plt.plot(m[0] / (2 * np.pi), F_chi_model_nonoise, c="C2", lw=1.5, alpha=0.85)
    if ray_id == '+7':
        plt.plot(m[0] / (2 * np.pi), F_chi_model_nonoise2, c="C2", linestyle='dashdot', lw=1.5, alpha=0.85)
    plt.plot(m_extend[0] / (2 * np.pi), F_chi_model, c="C2", lw=0.75, linestyle='dashed')
    
    # F_chi observed
    plt.plot(m_data, F_chi_lm[ray_id], c="C3", marker="o", markersize=3)
    plt.fill_between(
        m_data,
        F_chi_lm[ray_id] * F_chi_ls[ray_id],
        F_chi_lm[ray_id] / F_chi_ls[ray_id],
        alpha=0.3,
        color="C3",
        lw=0,
    )

    # Observational Noise Floor
    #plt.hlines([F_phi_obs_noise[ray_id], F_chi_obs_noise[ray_id]], 1e-2, 1e-0, colors=['C0', 'C2'])
    #print(F_phi_obs_noise[ray_id], F_chi_obs_noise[ray_id])
    
    plt.ylim([2e-4, 3e4])
    plt.xlim([2e-4, 4e-1])

    plt.xlabel("vertical wave number [1/m]")
    plt.ylabel("$F_{\\phi, \\chi}$ [$\\mathrm{rad} \\ m$]")

    plt.yscale("log")
    plt.xscale("log")

    plt.grid()

    plt.text(
        0.73,
        0.9,
        f"Ray ID: {ray_id}",
        transform=axes[k].transAxes,
        fontsize=9,
        bbox=dict(facecolor="white", edgecolor="k", pad=3, linewidth=0.5),
    )
    
    plt.text(
        0.04,
        0.07,
        f"{results[ray_id].x[0]:0.1f} $\\times$ GM Energy\n$F_{{\\phi}}$ noise level = {results[ray_id].x[1]:0.2f} [rad m]\n$F_{{\\chi}}$ noise level = {results[ray_id].x[2]:0.2f} [rad m]",
        transform=axes[k].transAxes,
        fontsize=8,
        bbox=dict(facecolor="white", edgecolor="k", pad=3, linewidth=0.5)
    )

plt.tight_layout()

#fig.savefig('../../figures/F_phi_chi_ducted.pdf', bbox_inches='tight')